# Reduced Data Pipeline

### Thie notebook is a reduced version of the main Data Pipeline Notebook. More precisely, this notebook does not compute any of the high cardinality data, however utilizes the full dataset. 

March 5th, 2025

Maxime Bouthillier

### Importing Libraries and Specialty Functions

In [15]:
import pandas as pd
import os 
import glob 
from datetime import datetime
import warnings
from joblib import dump
import Pipeline_Functions as func

cwd = os.getcwd()
print("Current working directory:", cwd)

Current working directory: /mnt/hpc/work/mbouthil/Research_Project/MIMIC-III_data


# Data Cleaning

### Admissions

In [16]:
# Setting the Directory
directory = '/mnt/hpc/work/mbouthil/Research_Project/MIMIC-III_data'
os.chdir(directory)


# Reading the csv file
adm_df = pd.read_csv("ADMISSIONS.csv")
adm_df.columns = adm_df.columns.str.lower()


# Cleaning the time based features
adm_df["edregtime"] = adm_df["edregtime"].fillna("1677-09-22 00:00:00")                                             
adm_df["edouttime"] = adm_df["edouttime"].fillna("1677-09-22 00:00:00")

adm_df = func.as_datetime(adm_df, column='admittime')
adm_df = func.as_datetime(adm_df, column='dischtime')
adm_df = func.as_datetime(adm_df, column='edregtime')
adm_df = func.as_datetime(adm_df, column='edouttime')  


# Removing all admission instances where a patient died
adm_df = adm_df.drop(adm_df[adm_df['hospital_expire_flag'] == 1].index)


# Setting Marital Status to binary variables
adm_df['marital_status'] = adm_df['marital_status'].apply(lambda x: "MARRIED" if x == "MARRIED" else "SINGLE")

In [17]:
# Creating Readmission Feature and subsetting the dataset
adm_df = func.readmission(adm_df, 30)

# Admission Duration time feature
adm_df['admit_duration'] = adm_df['dischtime'] - adm_df['admittime']

# ED Duration time feature
adm_df['ed_duration'] = adm_df['edouttime'] - adm_df['edregtime']


# Removing unnecessary variables
col_drops = ["row_id", "language", "religion", "hospital_expire_flag", "hadm_id", 
             "has_chartevents_data", "edregtime", "edouttime", "deathtime", "diagnosis"]

for i in col_drops:
    adm_df = adm_df.drop(i, axis=1)

# Checing NaN instances
func.check_nan(adm_df)

### Patients

In [18]:
# Reading the csv file
pat_df = pd.read_csv("PATIENTS.csv")
pat_df.columns = pat_df.columns.str.lower()
pat_df = func.as_datetime(pat_df, column='dob')

# Selecting only the necessary columns
pat_df = pat_df[['subject_id', 'gender', 'dob']]


# Double checking that there are no NaN values
func.check_nan(pat_df)

### ICU LOS

In [19]:
# Reading the csv file
icu_df = pd.read_csv("ICUSTAYS.csv")
icu_df.columns = icu_df.columns.str.lower()
icu_df = icu_df.dropna()


# Cleaning the time based features
icu_df  = func.as_datetime(icu_df , column='intime')
icu_df = func.as_datetime(icu_df , column='outtime')


# Subsetting the dataframe by only the releveat subject_ID entries:
icu_df = func.subject_subset(icu_df , adm_df, column='intime')

In [20]:
# Keeping only the necessary columns
icu_df  = icu_df[['subject_id', 'los']]

# Combining discontinuous ICU stays
icu_df = icu_df.groupby('subject_id', as_index=False).agg({'los': 'sum'})

# Double checking that there are no NaN values
func.check_nan(icu_df)

### Merging Dataframes

In [ ]:
# Left join of Patients table on Admission table
master_df = pd.merge(adm_df, pat_df, how='inner', on='subject_id')

# Left join of ICU LOS table on df
master_df = pd.merge(master_df, icu_df, how='inner', on='subject_id')

master_df = master_df.fillna(0)

# Creatinon of the Age variable and removal of the dob column
master_df['age'] = master_df['admittime'].dt.year - master_df['dob'].dt.year
master_df = master_df.drop('dob', axis=1)

# Dropping the unnecessary columns once combined
master_df = master_df.drop(['subject_id', 'admittime', 'dischtime'], axis=1)

dump(master_df, "Master_Dataframe_reduced.joblib")

['Master_Dataframe_reduced.joblib']